# 08 — Evaluating RAG with Ragas (and Storing Results in Couchbase)

Companion to [Chapter 13](../docs/13-evaluation-ragas.md). Couchbase has no built-in
eval framework — we use **Ragas**, and store every run in Couchbase so quality becomes
a queryable time series.

1. Build an eval set and run the notebook-03 pipeline over it
2. Score with the four core RAG metrics
3. Persist runs + per-sample scores to `ai.evals.*`
4. Regression analysis with SQL++

**Prerequisites:** notebooks 01–03 (corpus, index, RAG chain); `OPENAI_API_KEY`.

In [ ]:
%pip install -q couchbase python-dotenv ragas langchain-couchbase langchain-openai

In [ ]:
import os
import subprocess
from datetime import datetime, timezone, timedelta

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(os.getenv("ENV_FILE", ".env"), usecwd=True))  # ENV_FILE lets .env.server / .env.capella run side by side

from couchbase.auth import PasswordAuthenticator
from couchbase.cluster import Cluster
from couchbase.options import ClusterOptions, KnownConfigProfiles, QueryOptions

opts = ClusterOptions(PasswordAuthenticator(os.getenv("CB_USERNAME"),
                                            os.getenv("CB_PASSWORD")))
conn = os.getenv("CB_CONN_STRING")
if conn.startswith("couchbases://"):
    opts.apply_profile(KnownConfigProfiles.WanDevelopment)
cluster = Cluster.connect(conn, opts)
cluster.wait_until_ready(timedelta(seconds=10))

CB_BUCKET = os.getenv("CB_BUCKET")
bucket = cluster.bucket(CB_BUCKET)

## 1. The system under test — notebook 03's RAG pipeline, reassembled

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_couchbase.vectorstores import CouchbaseSearchVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

if os.getenv("CAPELLA_AI_ENDPOINT"):
    import base64
    key = os.getenv("CAPELLA_AI_TOKEN") or base64.b64encode(
        f"{os.environ['CB_USERNAME']}:{os.environ['CB_PASSWORD']}".encode()).decode()
    embeddings = OpenAIEmbeddings(openai_api_base=os.environ["CAPELLA_AI_ENDPOINT"],
                                  openai_api_key=key,
                                  model=os.getenv("CAPELLA_EMBEDDING_MODEL",
                                                  "intfloat/e5-mistral-7b-instruct"),
                                  check_embedding_ctx_length=False, tiktoken_enabled=False)
    llm = ChatOpenAI(openai_api_base=os.environ["CAPELLA_AI_ENDPOINT"], openai_api_key=key,
                     model=os.getenv("CAPELLA_LLM_MODEL", "meta-llama/Llama-3.1-8B-Instruct"),
                     temperature=0)
else:
    embeddings = OpenAIEmbeddings(model=os.getenv("EMBEDDING_MODEL", "text-embedding-3-small"))
    llm = ChatOpenAI(model=os.getenv("LLM_MODEL", "gpt-4o-mini"), temperature=0)

vector_store = CouchbaseSearchVectorStore(
    cluster=cluster, bucket_name=CB_BUCKET, scope_name="docs",
    collection_name="chunks", embedding=embeddings, index_name="chunks-vector-index",
)

answer_prompt = ChatPromptTemplate.from_template(
    "Answer using ONLY this context. If it's not in the context, say you don't know."
    "\n\nContext:\n{context}\n\nQuestion: {question}")
gen_chain = answer_prompt | llm | StrOutputParser()


def run_pipeline(question: str, k: int = 3) -> dict:
    """Run retrieval + generation, returning everything Ragas needs."""
    docs = vector_store.similarity_search(question, k=k)
    contexts = [d.page_content for d in docs]
    answer = gen_chain.invoke({"context": "\n\n---\n\n".join(contexts),
                               "question": question})
    return {"question": question, "contexts": contexts, "answer": answer}

## 2. The eval set

Question + reference pairs. Start with 20–50 written from *real user questions*; grow it
every time you find a failure. It's data — in a real project it lives in `ai.evals.cases`,
versioned like code. (Ours is small so the notebook runs cheaply.)

In [ ]:
EVAL_CASES = [
    {"question": "How do I rotate database credentials in Capella?",
     "reference": "Open Settings, choose Database Access, create a new credential, deploy "
                  "it to applications, then revoke the old one. Rotate at least quarterly."},
    {"question": "What settings does a vector field need in a Couchbase search index?",
     "reference": "dims matching the embedding model, similarity (dot_product or l2_norm), "
                  "and vector_index_optimized_for (recall or latency)."},
    {"question": "How should agent short-term memory expire?",
     "reference": "Session documents carry a TTL that slides on each interaction, so "
                  "inactive sessions expire automatically."},
    {"question": "How do you delete all memories for a user under GDPR?",
     "reference": "Run a SQL++ DELETE on the memories collection filtered by user_id."},
]

In [ ]:
from ragas import EvaluationDataset, SingleTurnSample

samples = []
for case in EVAL_CASES:
    out = run_pipeline(case["question"])
    samples.append(SingleTurnSample(
        user_input=out["question"],
        retrieved_contexts=out["contexts"],
        response=out["answer"],
        reference=case["reference"],
    ))
print(f"collected {len(samples)} samples")
print("example answer:", samples[0].response[:120], "…")

## 3. Score: the four metrics that triangulate failure

- `faithfulness` low → generation invents (fix prompt/model)
- `context_recall` low → retrieval misses (fix chunking/embeddings/hybrid)
- `context_precision` low → retrieval is noisy (fix k/filters)
- `answer_relevancy` low → answers dodge the question

Judge rules: strong model, pinned version, temperature 0.

In [ ]:
import warnings

# ragas 0.4 ships a new `ragas.metrics.collections` API (instructor-based LLMs, per-metric
# `score`/`ascore` calls) that isn't a drop-in replacement here: `evaluate()` still expects
# the legacy `Metric` class hierarchy passed via `metrics=[...]`, and `llm_factory` wants a
# raw OpenAI-style client rather than the `ChatOpenAI`/`LangchainLLMWrapper` combo we use to
# point at either OpenAI or a self-hosted Capella AI Services endpoint. Reworking this cell
# around the collections API would mean scoring each metric by hand instead of one
# `evaluate()` call, so we just silence the two deprecation warnings narrowly.
warnings.filterwarnings("ignore", category=DeprecationWarning, module="ragas.metrics")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="ragas.llms")

from ragas import evaluate, RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (answer_relevancy, context_precision, context_recall,
                           faithfulness)

# A stronger judge than the pipeline's own generator, where available. On Capella there's
# a single configured LLM, so gen and judge share it (Ch. 8 §8.5 — model choice is config).
judge = LangchainLLMWrapper(llm if os.getenv("CAPELLA_AI_ENDPOINT")
                            else ChatOpenAI(model="gpt-4o", temperature=0))

# Capella AI Services is a small, self-hosted deployment — the default RunConfig
# (max_workers=16, timeout=180s) fires 16 concurrent judge calls at it, which queues
# requests behind each other until most jobs blow past the timeout (observed: 121s/it,
# most metrics NaN from TimeoutError). Turning concurrency down lets each request actually
# get served instead of competing for the same limited capacity, and raising the timeout
# gives slow-but-alive requests room to finish rather than being killed mid-flight. Hosted
# OpenAI can handle far more parallelism, so keep it snappier there.
run_config = RunConfig(timeout=600, max_workers=2) if os.getenv("CAPELLA_AI_ENDPOINT") \
    else RunConfig(timeout=300, max_workers=8)

result = evaluate(
    dataset=EvaluationDataset(samples=samples),
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=judge,
    embeddings=embeddings,
    run_config=run_config,
)
result

## 4. Persist the run — quality as a time series

In [ ]:
def git_sha() -> str:
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       text=True).strip()
    except Exception:
        return "unknown"


runs_coll = bucket.scope("evals").collection("runs")
samples_coll = bucket.scope("evals").collection("samples")

df = result.to_pandas()
metric_cols = [c for c in df.columns
               if c not in ("user_input", "retrieved_contexts", "response", "reference")]

run_key = "evalrun::" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
runs_coll.upsert(run_key, {
    "type": "eval_run",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "git_commit": git_sha(),
    "pipeline": {"embedding_model": "text-embedding-3-small", "k": 3,
                 "chunking": "markdown-600-80", "llm": "gpt-4o-mini",
                 "judge": "gpt-4o"},
    "metrics": {m: float(df[m].mean()) for m in metric_cols},
    "n_samples": len(df),
})

for i, row in enumerate(df.to_dict("records")):
    samples_coll.upsert(f"{run_key}::sample::{i:04d}", {
        "type": "eval_sample", "run": run_key,
        "user_input": row["user_input"],
        "response": row["response"],
        "reference": row["reference"],
        **{m: (float(row[m]) if row[m] == row[m] else None) for m in metric_cols},  # NaN-safe
    })

print("stored", run_key, "with", len(df), "samples")

## 5. Regression analysis in SQL++

Aggregate scores say *that* something broke; per-sample storage says *what*.

In [ ]:
cluster.query(f"CREATE PRIMARY INDEX IF NOT EXISTS ON `{CB_BUCKET}`.evals.runs").execute()
cluster.query(f"CREATE PRIMARY INDEX IF NOT EXISTS ON `{CB_BUCKET}`.evals.samples").execute()

print("run history:")
for row in cluster.query(f"""
    SELECT r.created_at, r.git_commit, r.pipeline.embedding_model,
           ROUND(r.metrics.faithfulness, 3) AS faithfulness,
           ROUND(r.metrics.context_recall, 3) AS recall
    FROM `{CB_BUCKET}`.evals.runs r
    ORDER BY r.created_at DESC LIMIT 5"""):
    print(" ", row)

In [ ]:
# The debugging goldmine: which QUESTIONS scored worst this run?
for row in cluster.query(f"""
    SELECT s.user_input, ROUND(s.faithfulness, 3) AS faithfulness,
           ROUND(s.context_recall, 3) AS recall
    FROM `{CB_BUCKET}`.evals.samples s
    WHERE s.run = $run
    ORDER BY s.faithfulness ASC LIMIT 3""",
    QueryOptions(named_parameters={"run": run_key})):
    print(row)

Re-run this notebook after changing anything upstream — chunk size in 02, k or prompt in
03, the embedding model — and the runs table becomes your before/after evidence. Wire the
same code into CI as a PR gate (Ch. 13 §13.6: smoke set per-PR, full set nightly).

For **agent** evaluation (tool efficiency, response faithfulness from activity logs), see
`apps/support-agent/evals/` and Ragas' Agent Catalog integration
(`ragas.integrations.agentc`).

*This closes the loop the book opened in Chapter 1: ship → observe → evaluate → fix →
re-measure, with every byte of it — corpus, memory, catalogs, checkpoints, and now
quality scores — in one database.*